# Swiss Roll로 배우는 t-SNE / UMAP / Isomap / LLE 비교

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gshong-ai/ADA26/blob/claude/mnist-dimensionality-reduction-H2Meg/03_manifold_learning/swiss_roll_manifold.ipynb)

## 개요

Swiss Roll은 **정답을 아는 매니폴드** 입니다.  
매니폴드 파라미터 `t`(나선 위치)의 색상이 2D 투영에서 **무지개처럼 연속적**이면 성공입니다.

| 기법 | 핵심 아이디어 | 주요 파라미터 | 속도 |
|------|-------------|------------|:----:|
| **Isomap** | k-NN 그래프 최단 경로 → 측지 거리 보존 | `n_neighbors` | 중간 |
| **LLE** | 이웃의 선형 재구성 가중치 보존 | `n_neighbors`, `method` | 빠름 |
| **t-SNE** | KL 발산으로 근방 확률 분포 매칭 | `perplexity` | 느림 |
| **UMAP** | 리만 기하 기반 위상 구조 보존 | `n_neighbors`, `min_dist` | 빠름 |

### 이 노트북의 목표
1. 각 기법으로 Swiss Roll을 어떻게 펼치는지 직접 확인
2. 주요 하이퍼파라미터가 결과에 미치는 영향 시각화
3. 계산 속도 + 정량 지표(연속성·신뢰성·실루엣)로 객관적 비교

## 0. 패키지 설치

In [ ]:
!pip install -q plotly scikit-learn umap-learn

## 1. 라이브러리 임포트

In [ ]:
# ─── 라이브러리 임포트 ───
import numpy as np
import time

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.datasets import make_swiss_roll
from sklearn.manifold import Isomap, LocallyLinearEmbedding, TSNE, trustworthiness
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

import umap

pio.renderers.default = 'colab'
print(f'scikit-learn 로드 완료')
print(f'umap-learn 로드 완료')

## 2. 하이퍼파라미터 & 설정

In [ ]:
# ─── 공통 설정 ───
RANDOM_SEED      = 42
N_SAMPLES        = 1500     # Swiss Roll 샘플 수
SWISS_NOISE      = 0.1      # 노이즈 (작게 → 구조 선명)
N_T_CLASSES      = 6        # 실루엣 스코어용 t 등분 클래스 수

# ─── 각 기법 기본 파라미터 (종합 비교에서 사용) ───
ISOMAP_N_NEIGHBORS = 10
LLE_N_NEIGHBORS    = 12
LLE_METHOD         = 'standard'
TSNE_PERPLEXITY    = 30
TSNE_N_ITER        = 1000
UMAP_N_NEIGHBORS   = 15
UMAP_MIN_DIST      = 0.1

# ─── 민감도 분석 범위 ───
ISOMAP_K_RANGE   = [5, 10, 15, 20]
LLE_METHODS      = ['standard', 'modified', 'ltsa']
LLE_K_RANGE      = [8, 12, 20]
TSNE_PERP_RANGE  = [5, 30, 50, 100]
UMAP_K_RANGE     = [5, 15, 30]
UMAP_DIST_RANGE  = [0.05, 0.1, 0.5]

# ─── 시각화 ───
COLORSCALE       = 'Rainbow'
OUTPUT_HTML      = 'swiss_roll_manifold.html'

np.random.seed(RANDOM_SEED)
print('설정 완료')

## 3. 데이터 로드 & 전처리

`t`: 나선 위치 파라미터 — **이 색상이 2D에서 무지개처럼 연속적이면 언폴딩 성공**

In [ ]:
# ─── Swiss Roll 생성 ───
X_raw, t = make_swiss_roll(n_samples=N_SAMPLES, noise=SWISS_NOISE,
                            random_state=RANDOM_SEED)

scaler = StandardScaler()
X = scaler.fit_transform(X_raw).astype('float32')

# 평가용 클래스 레이블 (t를 N_T_CLASSES 등분)
bins  = np.percentile(t, np.linspace(0, 100, N_T_CLASSES + 1)[1:-1])
y_cls = np.digitize(t, bins=bins)

print(f'Swiss Roll shape : {X.shape}')
print(f't 범위           : {t.min():.2f} ~ {t.max():.2f}')
print(f'클래스 분포      : {np.bincount(y_cls).tolist()}')

# ─── 공통 scatter trace 생성 함수 ───
def scatter2d(coords, t_vals, show_colorbar=False):
    """t로 색상 입힌 2D 산점도 trace"""
    return go.Scatter(
        x=coords[:, 0], y=coords[:, 1],
        mode='markers',
        marker=dict(
            size=3, color=t_vals, colorscale=COLORSCALE, opacity=0.8,
            showscale=show_colorbar,
            colorbar=dict(title='t') if show_colorbar else None,
        ),
        showlegend=False,
    )

# ─── 3D 원본 시각화 ───
fig_3d = go.Figure(go.Scatter3d(
    x=X_raw[:, 0], y=X_raw[:, 1], z=X_raw[:, 2],
    mode='markers',
    marker=dict(size=3, color=t, colorscale=COLORSCALE, opacity=0.85,
                colorbar=dict(title='t (매니폴드 위치)')),
))
fig_3d.update_layout(
    title='원본 Swiss Roll (3D) — t 색상이 연속적: 매니폴드 파라미터 위치',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    width=700, height=520,
)
fig_3d.show()

## 4. Isomap — 측지 거리 기반 MDS

### 알고리즘
```
[1단계] k-NN 그래프 구축   : 각 점에서 k개 이웃을 연결 (엣지 = 유클리드 거리)
[2단계] 최단 경로 계산     : Dijkstra → 측지 거리 행렬 D_geo
[3단계] 고전 MDS           : D_geo로 2D 투영
```

### `n_neighbors`의 영향
- **너무 작으면** → 그래프 단절 → 측지 거리 = ∞  
- **너무 크면** → 다른 나선 층 간에 "지름길"이 생겨 측지 거리 과소 추정

In [ ]:
# ─── Isomap 기본 파라미터 적용 ───
print(f'Isomap 계산 중... (n_neighbors={ISOMAP_N_NEIGHBORS})')
t0 = time.perf_counter()
isomap_main = Isomap(n_components=2, n_neighbors=ISOMAP_N_NEIGHBORS)
coords_isomap = isomap_main.fit_transform(X)
time_isomap = time.perf_counter() - t0
print(f'완료 ({time_isomap:.2f}초)  재구성 오차: {isomap_main.reconstruction_error():.4f}')

# ─── n_neighbors 민감도 분석 ───
print('\nn_neighbors 민감도 분석 중...')
iso_coords_list = []
iso_errs = []
for k in ISOMAP_K_RANGE:
    iso_k = Isomap(n_components=2, n_neighbors=k)
    c = iso_k.fit_transform(X)
    iso_coords_list.append(c)
    iso_errs.append(iso_k.reconstruction_error())
    print(f'  n_neighbors={k:2d}: 재구성 오차={iso_k.reconstruction_error():.4f}')

fig_iso_k = make_subplots(
    rows=1, cols=len(ISOMAP_K_RANGE),
    subplot_titles=[f'k={k}\n(오차={e:.3f})' for k, e in zip(ISOMAP_K_RANGE, iso_errs)],
)
for col, (k, coords) in enumerate(zip(ISOMAP_K_RANGE, iso_coords_list), 1):
    show_bar = (col == len(ISOMAP_K_RANGE))
    fig_iso_k.add_trace(scatter2d(coords, t, show_colorbar=show_bar), row=1, col=col)

fig_iso_k.update_layout(
    title='Isomap — n_neighbors 민감도 (작으면 그래프 단절, 크면 지름길 발생)',
    width=1100, height=310,
)
fig_iso_k.show()

## 5. LLE — 이웃의 선형 재구성 가중치 보존

### 알고리즘
```
[1단계] 이웃 재구성 가중치 학습:
        각 점 xᵢ를 k개 이웃의 선형 결합으로 표현: min Σ ‖xᵢ - Σⱼ Wᵢⱼ xⱼ‖²

[2단계] 저차원 임베딩:
        같은 가중치 W로 2D에서도 재구성: min Σ ‖yᵢ - Σⱼ Wᵢⱼ yⱼ‖²
        → 희소 고유값 문제
```

### LLE 변형 (`method`)
| 방법 | 특징 |
|------|------|
| `standard` | 기본 LLE — 경계 근처에서 불안정할 수 있음 |
| `modified` | 공분산 정규화 추가 — 더 안정적 |
| `ltsa` | 국소 접선 공간 정렬 — Swiss Roll에 특히 효과적 |

In [ ]:
# ─── LLE 기본 파라미터 적용 ───
print(f'LLE 계산 중... (method={LLE_METHOD}, n_neighbors={LLE_N_NEIGHBORS})')
t0 = time.perf_counter()
lle_main = LocallyLinearEmbedding(n_components=2, n_neighbors=LLE_N_NEIGHBORS,
                                   method=LLE_METHOD, random_state=RANDOM_SEED)
coords_lle = lle_main.fit_transform(X)
time_lle = time.perf_counter() - t0
print(f'완료 ({time_lle:.2f}초)  재구성 오차: {lle_main.reconstruction_error_:.6f}')

# ─── 방법(method) 비교 ───
print('\nLLE 방법 비교 중...')
lle_method_coords = []
lle_method_errs   = []
for method in LLE_METHODS:
    lle_m = LocallyLinearEmbedding(n_components=2, n_neighbors=LLE_N_NEIGHBORS,
                                    method=method, random_state=RANDOM_SEED)
    try:
        c = lle_m.fit_transform(X)
        err = lle_m.reconstruction_error_
    except Exception as e:
        print(f'  {method}: 오류 → {e}')
        c = np.zeros((N_SAMPLES, 2))
        err = float('nan')
    lle_method_coords.append(c)
    lle_method_errs.append(err)
    print(f'  method={method:8s}: 재구성 오차={err:.6f}')

fig_lle_m = make_subplots(
    rows=1, cols=len(LLE_METHODS),
    subplot_titles=[f'{m}\n(오차={e:.5f})' for m, e in zip(LLE_METHODS, lle_method_errs)],
)
for col, (method, coords) in enumerate(zip(LLE_METHODS, lle_method_coords), 1):
    fig_lle_m.add_trace(scatter2d(coords, t, show_colorbar=(col == len(LLE_METHODS))),
                        row=1, col=col)

fig_lle_m.update_layout(
    title='LLE — method 비교 (standard / modified / ltsa)',
    width=950, height=330,
)
fig_lle_m.show()

# ─── n_neighbors 민감도 분석 (standard) ───
print('\nLLE n_neighbors 민감도 분석 중...')
lle_k_coords = []
for k in LLE_K_RANGE:
    lle_k = LocallyLinearEmbedding(n_components=2, n_neighbors=k,
                                    method=LLE_METHOD, random_state=RANDOM_SEED)
    lle_k_coords.append(lle_k.fit_transform(X))
    print(f'  n_neighbors={k}: 완료')

fig_lle_k = make_subplots(
    rows=1, cols=len(LLE_K_RANGE),
    subplot_titles=[f'k={k}' for k in LLE_K_RANGE],
)
for col, (k, coords) in enumerate(zip(LLE_K_RANGE, lle_k_coords), 1):
    fig_lle_k.add_trace(scatter2d(coords, t, show_colorbar=(col == len(LLE_K_RANGE))),
                        row=1, col=col)

fig_lle_k.update_layout(
    title=f'LLE (standard) — n_neighbors 민감도',
    width=900, height=310,
)
fig_lle_k.show()

## 6. t-SNE — KL 발산으로 근방 확률 분포 매칭

### 알고리즘
```
고차원 유사도: pᵢⱼ = 가우시안 커널로 정의한 이웃 확률 (perplexity로 밴드폭 조정)
저차원 유사도: qᵢⱼ = 스튜던트 t 분포 (자유도 1) — 코시 커널

목적: KL(P ‖ Q) 최소화 → 고차원 이웃 구조를 2D에서 재현
```

### `perplexity`의 의미
각 점이 "실질적으로 몇 개의 이웃을 가지느냐"를 결정합니다.

```
perplexity ≈ 2^H(P)  (H: 조건부 엔트로피)

낮은 perplexity (5~10)  → 아주 국소적 구조만 보존, 전역 구조 왜곡
중간 perplexity (20~50) → 일반적으로 권장 범위
높은 perplexity (100+)  → 전역 구조 더 반영, 클러스터 경계 흐릿
```

> ⚠️ t-SNE는 **전역 거리 보존을 보장하지 않습니다**.  
> 클러스터 사이의 거리, 클러스터 크기는 해석하지 마세요.

In [ ]:
# ─── t-SNE 기본 파라미터 적용 ───
print(f't-SNE 계산 중... (perplexity={TSNE_PERPLEXITY})')
t0 = time.perf_counter()
tsne_main = TSNE(n_components=2, perplexity=TSNE_PERPLEXITY,
                  n_iter=TSNE_N_ITER, random_state=RANDOM_SEED)
coords_tsne = tsne_main.fit_transform(X)
time_tsne = time.perf_counter() - t0
print(f'완료 ({time_tsne:.2f}초)  KL 발산: {tsne_main.kl_divergence_:.4f}')

# ─── perplexity 민감도 분석 ───
print('\nperplexity 민감도 분석 중 (시간이 걸립니다)...')
perp_coords = []
perp_kl     = []
for perp in TSNE_PERP_RANGE:
    ts = TSNE(n_components=2, perplexity=perp, n_iter=TSNE_N_ITER,
               random_state=RANDOM_SEED)
    c = ts.fit_transform(X)
    perp_coords.append(c)
    perp_kl.append(ts.kl_divergence_)
    print(f'  perplexity={perp:3d}: KL={ts.kl_divergence_:.4f}')

# 2×2 그리드
fig_perp = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'perplexity={p}  (KL={kl:.3f})'
        for p, kl in zip(TSNE_PERP_RANGE, perp_kl)
    ],
    horizontal_spacing=0.08, vertical_spacing=0.14,
)
positions = [(1,1),(1,2),(2,1),(2,2)]
for (p, coords, kl), (row, col) in zip(
    zip(TSNE_PERP_RANGE, perp_coords, perp_kl), positions
):
    show_bar = (row == 1 and col == 2)
    fig_perp.add_trace(scatter2d(coords, t, show_colorbar=show_bar), row=row, col=col)

fig_perp.update_layout(
    title='t-SNE — perplexity 민감도<br>'
          '<sup>작으면: 파편화 / 크면: 전역 구조 반영 증가, but 클러스터 경계 흐릿</sup>',
    width=920, height=650,
)
fig_perp.show()

## 7. UMAP — 리만 기하 기반 위상 구조 보존

### 알고리즘 (직관적 요약)
```
[1단계] 퍼지 단체 복합체(Fuzzy Simplicial Complex) 구축
        각 점에서 n_neighbors 이웃을 리만 거리로 연결
        → 고차원 위상 구조 표현

[2단계] 저차원 임베딩 최적화
        cross-entropy로 고차원/저차원 퍼지 집합 매칭
        → 위상 구조 보존하며 2D 배치
```

### 주요 파라미터

| 파라미터 | 역할 | 권장 범위 |
|---------|------|----------|
| `n_neighbors` | 국소 vs 전역 균형 — 클수록 전역 구조 강조 | 5 ~ 50 |
| `min_dist` | 2D에서 점들을 얼마나 밀집시킬지 — 클수록 퍼짐 | 0.0 ~ 0.99 |

> t-SNE와 달리 **전역 구조도 어느 정도 보존**되며 속도가 훨씬 빠릅니다.

In [ ]:
# ─── UMAP 기본 파라미터 적용 ───
print(f'UMAP 계산 중... (n_neighbors={UMAP_N_NEIGHBORS}, min_dist={UMAP_MIN_DIST})')
t0 = time.perf_counter()
umap_main = umap.UMAP(n_components=2, n_neighbors=UMAP_N_NEIGHBORS,
                       min_dist=UMAP_MIN_DIST, random_state=RANDOM_SEED)
coords_umap = umap_main.fit_transform(X)
time_umap = time.perf_counter() - t0
print(f'완료 ({time_umap:.2f}초)')

# ─── n_neighbors 민감도 분석 ───
print('\nn_neighbors 민감도 분석 중...')
umap_k_coords = []
for k in UMAP_K_RANGE:
    um = umap.UMAP(n_components=2, n_neighbors=k,
                    min_dist=UMAP_MIN_DIST, random_state=RANDOM_SEED)
    umap_k_coords.append(um.fit_transform(X))
    print(f'  n_neighbors={k:2d}: 완료')

fig_uk = make_subplots(
    rows=1, cols=len(UMAP_K_RANGE),
    subplot_titles=[f'n_neighbors={k}' for k in UMAP_K_RANGE],
)
for col, (k, coords) in enumerate(zip(UMAP_K_RANGE, umap_k_coords), 1):
    fig_uk.add_trace(scatter2d(coords, t, show_colorbar=(col == len(UMAP_K_RANGE))),
                     row=1, col=col)
fig_uk.update_layout(
    title=f'UMAP — n_neighbors 민감도 (min_dist={UMAP_MIN_DIST} 고정)'
          '<br><sup>작으면: 아주 국소적 구조 / 크면: 전역 구조 반영 증가</sup>',
    width=950, height=330,
)
fig_uk.show()

# ─── min_dist 민감도 분석 ───
print('\nmin_dist 민감도 분석 중...')
umap_d_coords = []
for d in UMAP_DIST_RANGE:
    um = umap.UMAP(n_components=2, n_neighbors=UMAP_N_NEIGHBORS,
                    min_dist=d, random_state=RANDOM_SEED)
    umap_d_coords.append(um.fit_transform(X))
    print(f'  min_dist={d}: 완료')

fig_ud = make_subplots(
    rows=1, cols=len(UMAP_DIST_RANGE),
    subplot_titles=[f'min_dist={d}' for d in UMAP_DIST_RANGE],
)
for col, (d, coords) in enumerate(zip(UMAP_DIST_RANGE, umap_d_coords), 1):
    fig_ud.add_trace(scatter2d(coords, t, show_colorbar=(col == len(UMAP_DIST_RANGE))),
                     row=1, col=col)
fig_ud.update_layout(
    title=f'UMAP — min_dist 민감도 (n_neighbors={UMAP_N_NEIGHBORS} 고정)'
          '<br><sup>작으면: 밀집 클러스터 / 크면: 균일하게 퍼짐</sup>',
    width=950, height=330,
)
fig_ud.show()

## 8. 종합 비교 시각화

각 기법의 **최적 파라미터** 결과를 나란히 비교합니다.

In [ ]:
# ─── 2×2 종합 비교 그리드 ───
plot_items = [
    (f'Isomap\n(k={ISOMAP_N_NEIGHBORS})',                           coords_isomap),
    (f'LLE [{LLE_METHOD}]\n(k={LLE_N_NEIGHBORS})',                  coords_lle),
    (f't-SNE\n(perplexity={TSNE_PERPLEXITY})',                      coords_tsne),
    (f'UMAP\n(k={UMAP_N_NEIGHBORS}, min_dist={UMAP_MIN_DIST})',     coords_umap),
]

fig_all = make_subplots(
    rows=2, cols=2,
    subplot_titles=[label.replace('\n', ' ') for label, _ in plot_items],
    horizontal_spacing=0.08,
    vertical_spacing=0.14,
)

for idx, (label, coords) in enumerate(plot_items):
    row, col = divmod(idx, 2)
    row += 1; col += 1
    show_bar = (row == 1 and col == 2)
    fig_all.add_trace(
        scatter2d(coords, t, show_colorbar=show_bar),
        row=row, col=col,
    )

fig_all.update_layout(
    title=dict(
        text='Swiss Roll 2D 투영 종합 비교 — 색상(t)이 무지개처럼 연속적일수록 언폴딩 성공',
        x=0.5, font=dict(size=15),
    ),
    width=1000, height=820,
    paper_bgcolor='white',
)
fig_all.show()

fig_all.write_html(OUTPUT_HTML, include_plotlyjs='cdn')
print(f'저장 완료: {OUTPUT_HTML}')

## 9. 계산 속도 비교

동일한 파라미터 조건에서 각 기법의 wall-clock time을 측정합니다.

In [ ]:
# ─── 계산 속도 벤치마크 ───
bench_methods = [
    ('Isomap',
     lambda: Isomap(n_components=2,
                    n_neighbors=ISOMAP_N_NEIGHBORS).fit_transform(X)),
    ('LLE',
     lambda: LocallyLinearEmbedding(n_components=2,
                                     n_neighbors=LLE_N_NEIGHBORS,
                                     method=LLE_METHOD,
                                     random_state=RANDOM_SEED).fit_transform(X)),
    ('t-SNE',
     lambda: TSNE(n_components=2, perplexity=TSNE_PERPLEXITY,
                   n_iter=TSNE_N_ITER,
                   random_state=RANDOM_SEED).fit_transform(X)),
    ('UMAP',
     lambda: umap.UMAP(n_components=2,
                        n_neighbors=UMAP_N_NEIGHBORS,
                        min_dist=UMAP_MIN_DIST,
                        random_state=RANDOM_SEED).fit_transform(X)),
]

print(f'=== 계산 속도 벤치마크 (N={N_SAMPLES}) ===')
times_dict = {}
for name, fn in bench_methods:
    t_start = time.perf_counter()
    fn()
    elapsed = time.perf_counter() - t_start
    times_dict[name] = elapsed
    print(f'  {name:8s}: {elapsed:6.2f}초')

# 막대 차트
colors_speed = ['#2ECC71', '#3498DB', '#E74C3C', '#9B59B6']
fig_speed = go.Figure(go.Bar(
    x=list(times_dict.keys()),
    y=list(times_dict.values()),
    marker_color=colors_speed,
    opacity=0.85,
    text=[f'{v:.2f}초' for v in times_dict.values()],
    textposition='auto',
))
fig_speed.update_layout(
    title=f'계산 속도 비교 (N={N_SAMPLES}, 단위: 초, 낮을수록 빠름)',
    yaxis_title='소요 시간 (초)',
    yaxis_type='log',
    width=650, height=420,
)
fig_speed.show()

fastest = min(times_dict, key=times_dict.get)
print(f'\n가장 빠른 기법: {fastest} ({times_dict[fastest]:.2f}초)')

## 10. 정량 평가

### 평가 지표 3가지

| 지표 | 의미 | 좋은 값 |
|------|------|--------|
| **이웃 t값 분산** | 2D 이웃들이 매니폴드에서도 이웃인가? | ↓ 낮을수록 |
| **Trustworthiness** | 2D에서 가깝지만 3D에서는 멀었던 점이 있는가? | ↑ 1에 가까울수록 |
| **실루엣 스코어** | t 등분 클래스가 2D에서 얼마나 잘 분리됐는가? | ↑ 1에 가까울수록 |

In [ ]:
# ─── 이웃 t값 분산 계산 ───
def continuity_score(coords_2d, t_vals, k=10):
    """k-이웃 t값 분산 평균 (낮을수록 매니폴드 구조 잘 보존)"""
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords_2d)
    _, indices = nbrs.kneighbors(coords_2d)
    return np.mean([np.var(t_vals[indices[i, 1:]]) for i in range(len(coords_2d))])

# ─── 전체 평가 ───
eval_items = [
    ('Isomap',  coords_isomap, '#2ECC71'),
    ('LLE',     coords_lle,    '#3498DB'),
    ('t-SNE',   coords_tsne,   '#E74C3C'),
    ('UMAP',    coords_umap,   '#9B59B6'),
]

print('=== 정량 평가 결과 ===')
print(f'{"기법":8s}  {"이웃 t 분산":>14s}  {"Trustworthiness":>16s}  {"실루엣":>10s}')
print('-' * 58)

cont_scores   = []
trust_scores  = []
sil_scores    = []
for name, coords, _ in eval_items:
    cont  = continuity_score(coords, t, k=10)
    trust = trustworthiness(X, coords, n_neighbors=10)
    sil   = silhouette_score(coords, y_cls)
    cont_scores.append(cont)
    trust_scores.append(trust)
    sil_scores.append(sil)
    print(f'{name:8s}  {cont:14.4f}  {trust:16.4f}  {sil:10.4f}')

# ─── 3개 지표 비교 막대 차트 ───
method_names = [name for name, _, _ in eval_items]
bar_colors   = [color for _, _, color in eval_items]

fig_eval = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '이웃 t값 분산 (↓ 낮을수록)',
        'Trustworthiness (↑ 높을수록)',
        '실루엣 스코어 (↑ 높을수록)',
    ],
)

for col, (scores, reverse) in enumerate(
    [(cont_scores, True), (trust_scores, False), (sil_scores, False)], 1
):
    fig_eval.add_trace(go.Bar(
        x=method_names, y=scores,
        marker_color=bar_colors, opacity=0.85,
        text=[f'{s:.4f}' for s in scores], textposition='auto',
        showlegend=False,
    ), row=1, col=col)

fig_eval.update_layout(
    title='Swiss Roll 2D 투영 정량 평가 — 3가지 지표 비교',
    width=1100, height=400,
)
fig_eval.show()

print(f'\n이웃 t 분산 최소 (최고): {eval_items[int(np.argmin(cont_scores))][0]}')
print(f'Trustworthiness 최고  : {eval_items[int(np.argmax(trust_scores))][0]}')
print(f'실루엣 스코어 최고    : {eval_items[int(np.argmax(sil_scores))][0]}')

## 11. 결론

### 기법별 Swiss Roll 분석 요약

| 기법 | 언폴딩 | 속도 | 재현성 | 주의사항 |
|------|:------:|:----:|:------:|----------|
| **Isomap** | ✓ 우수 | 중간 | ✓ 결정적 | k가 너무 크면 지름길 발생 |
| **LLE** | △ 방법 의존 | 빠름 | ✓ 결정적 | `ltsa`가 Swiss Roll에 가장 효과적 |
| **t-SNE** | ✓ 우수 | 느림 | ✗ 비결정적 | 전역 거리 보존 안 됨, perplexity 민감 |
| **UMAP** | ✓ 우수 | 빠름 | △ seed 고정 필요 | 전반적 균형 최고 |

### 하이퍼파라미터 핵심 정리

```
Isomap  n_neighbors : 너무 작으면 그래프 단절 / 너무 크면 지름길 발생
                       → Swiss Roll: 10~15 권장

LLE     method      : ltsa > modified > standard (Swiss Roll 기준)
        n_neighbors : 10~15 범위가 일반적으로 안정적

t-SNE   perplexity  : 5 → 파편화 / 30~50 → 권장 / 100+ → 전역 구조 혼합
                       → 절대적 거리·크기 해석 금지

UMAP    n_neighbors : 작으면 극단적 국소 / 크면 전역 구조 강조
        min_dist    : 작으면 촘촘한 클러스터 / 크면 균일한 퍼짐
                       → Swiss Roll: n_neighbors=15, min_dist=0.1 권장
```

### 핵심 교훈

> **매니폴드 학습에서 "어떤 이웃을 얼마나 믿느냐"가 성패를 결정합니다.**
>
> - Isomap / LLE: **그래프 구조**로 이웃 정의
> - t-SNE: **확률 분포**로 이웃 정의  
> - UMAP: **퍼지 집합(Fuzzy Set)**으로 이웃 정의
>
> Swiss Roll처럼 정답이 있는 데이터에서는 모두 작동하지만,
> 실전 데이터에서는 도메인 지식으로 파라미터를 조정해야 합니다.

### 더 나아가기
- **`04_autoencoder_manifold/`**: 신경망 기반 비선형 차원 축소
- **Parametric UMAP**: 새 데이터를 학습 없이 투영 가능한 UMAP 변형
- **Topological Data Analysis (TDA)**: 퍼시스턴트 호몰로지로 매니폴드 위상 분석